In [ ]:
import sys
print(sys.executable)

In [ ]:
import geopandas as gpd
print(gpd.__version__)
import shapely
print(shapely.__version__)

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("data/raw/burnaby/Burnaby_Boundary.geojson")
print(gdf.crs)

In [ ]:
import geopandas as gpd

da = gpd.read_file(
    "data/raw/population/lda_000b21a_e.zip"
)

burnaby = gpd.read_file(
    "data/raw/burnaby/Burnaby_Boundary.geojson"
)

burnaby = burnaby.to_crs(da.crs)

da_bc = da[da["PRUID"] == "59"].copy()

burnaby_geometry = burnaby.geometry.union_all()

inside = (
    da_bc.geometry
    .representative_point()
    .within(burnaby_geometry)
)

burnaby_da = da_bc.loc[inside].copy()

burnaby_dguids = (
    burnaby_da["DGUID"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Burnaby DA:", len(burnaby_da))
print("DGUID:", len(burnaby_dguids))
print(burnaby_dguids[:5])

In [ ]:
from urllib.request import Request, urlopen
from io import BytesIO
import pandas as pd

test_dguid = burnaby_dguids[0]

base_url = (
    "https://api.statcan.gc.ca/"
    "census-recensement/profile/sdmx/rest/data/"
    "STC_CP,DF_DA,1.3"
)

url = f"{base_url}/A5.{test_dguid}.1.1.1"

request = Request(
    url,
    headers={"Accept": "text/csv"}
)

with urlopen(request, timeout=30) as response:
    data = response.read()

test_population = pd.read_csv(BytesIO(data))

print(
    test_population[
        ["REF_AREA", "ALT_GEO_CODE", "OBS_VALUE", "FLAG"]
    ]
)

In [ ]:
import time
from pathlib import Path

batch_size = 25
results = []

for start in range(0, len(burnaby_dguids), batch_size):
    batch = burnaby_dguids[start:start + batch_size]
    dguid_parameter = "+".join(batch)

    batch_url = (
        f"{base_url}/"
        f"A5.{dguid_parameter}.1.1.1"
    )

    request = Request(
        batch_url,
        headers={
            "Accept": "text/csv",
            "User-Agent": "gis-ingest-tracker/1.0",
        },
    )

    with urlopen(request, timeout=30) as response:
        data = response.read()

    batch_result = pd.read_csv(
        BytesIO(data),
        dtype={
            "REF_AREA": "string",
            "ALT_GEO_CODE": "string",
            "FLAG": "string",
        },
    )

    results.append(batch_result)

    print(
        f"Downloaded batch "
        f"{start // batch_size + 1}: "
        f"{len(batch_result)} records"
    )

    time.sleep(0.2)

population_raw = pd.concat(
    results,
    ignore_index=True,
)

print("Total records:", len(population_raw))
print("Unique DGUID:", population_raw["REF_AREA"].nunique())
print("Missing population:", population_raw["OBS_VALUE"].isna().sum())

In [ ]:
from pathlib import Path

output_path = Path(
    "data/raw/population/"
    "burnaby_da_population_2021_api.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

population_raw.to_csv(
    output_path,
    index=False,
)

print("Saved:", output_path)